# Phase 4: Exploratory Data Analysis (EDA)
## Flight Delays Are Contagious: Predicting Delay Propagation Across Consecutive Aircraft Operations

**Research Objective:** Investigate the extent to which delays from preceding flight operations propagate to subsequent flights operated by the same aircraft.

**Dataset:** `data/processed/flight_rotation_pairs.parquet`  
**Sample Size:** 6,768,487 validated consecutive flight pairs  
**Time Horizon:** Full Year 2025 (BTS On-Time Performance)  
**Targets:** Classification (`NEXT_ARR_DEL15`), Regression (`NEXT_ARR_DELAY`)


In [ ]:
import os
import sys
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Ensure writable matplotlib cache
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib"
plt.style.use("seaborn-v0_8-whitegrid")

# Connect to DuckDB
con = duckdb.connect()
DATA_PATH = "data/processed/flight_rotation_pairs.parquet"
print(f"DuckDB version: {duckdb.__version__}")
print(f"Connected to dataset: {DATA_PATH}")


---
## 1. Target Variable Analysis

We investigate the distributional characteristics of both prediction targets:
1. **Binary Classification Target (`NEXT_ARR_DEL15`):** Standard FAA/BTS definition of flight arrival delay (1 if arrival delay $\ge 15$ minutes, 0 otherwise).
2. **Continuous Regression Target (`NEXT_ARR_DELAY`):** Difference between actual gate arrival and scheduled gate arrival in minutes (negative values indicate early arrivals).


In [ ]:
# Query target summary metrics from Parquet
q_targets = f"""
SELECT 
    COUNT(*) AS total_pairs,
    SUM(NEXT_ARR_DEL15) AS count_del15,
    SUM(CASE WHEN NEXT_ARR_DEL15 = 0 THEN 1 ELSE 0 END) AS count_ontime,
    ROUND(AVG(NEXT_ARR_DEL15) * 100.0, 2) AS pct_del15,
    ROUND(AVG(NEXT_ARR_DELAY), 2) AS mean_arr_delay,
    ROUND(MEDIAN(NEXT_ARR_DELAY), 2) AS median_arr_delay,
    ROUND(STDDEV(NEXT_ARR_DELAY), 2) AS std_arr_delay,
    ROUND(MIN(NEXT_ARR_DELAY), 2) AS min_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.25), 2) AS q25_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.75), 2) AS q75_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.90), 2) AS p90_arr_delay,
    ROUND(QUANTILE_CONT(NEXT_ARR_DELAY, 0.99), 2) AS p99_arr_delay,
    ROUND(MAX(NEXT_ARR_DELAY), 2) AS max_arr_delay
FROM read_parquet('{DATA_PATH}')
"""
df_targets = con.execute(q_targets).df()
display(df_targets)

# Display publication figure
display(Image(filename="results/figures/fig01_target_distributions.png"))


### Analytical Interpretation: Target Distributions & Skewness

1. **Class Imbalance (Classification):**
   - The overall delay rate is **22.02%** ($1,490,586$ delayed vs $5,277,901$ on-time flights), representing an imbalance ratio of **3.54:1**.
   - While moderate, this imbalance requires threshold-independent evaluation metrics (such as PR-AUC / Average Precision, ROC-AUC, and F1-score) rather than raw accuracy.
2. **Severe Right Skewness (Regression):**
   - The continuous delay distribution exhibits classic heavy-tailed, asymmetric right skew:
     - **Median Arrival Delay:** $-6.00$ minutes (over 58% of flights arrive early due to scheduled block-time padding).
     - **Mean Arrival Delay:** $+7.78$ minutes (pulled upward by severe positive outliers).
     - **Interquartile Range (IQR):** $[-15.0, +11.0]$ minutes (a 26-minute central spread).
     - **90th Percentile (P90):** $+47.0$ minutes | **99th Percentile (P99):** $+214.0$ minutes | **Maximum:** $+4,336.0$ minutes.
3. **Implications for Machine Learning:**
   - Standard Mean Squared Error (MSE) loss is hypersensitive to extreme multi-hour delays, heavily penalizing typical operations.
   - Robust evaluation metrics such as **Mean Absolute Error (MAE)** and **Median Absolute Error (MedAE)** must be prioritized in Phase 6.


---
## 2. Previous-Flight Delay Propagation Dynamics

**Analytical Question:** How strongly does an aircraft's preceding arrival delay condition the likelihood and severity of delay on its next scheduled flight?


In [ ]:
# Display delay propagation across prior delay intervals
df_bins = pd.read_csv("results/tables/eda_delay_propagation_bins.csv")
display(df_bins)

# Display transition matrix
df_trans = pd.read_csv("results/tables/eda_transition_matrix.csv")
display(df_trans)

# Display publication figure
display(Image(filename="results/figures/fig02_delay_propagation_relationships.png"))


### Analytical Interpretation: Delay Propagation Dynamics

1. **Non-Linear Threshold Propagation:**
   - When the preceding flight is early ($<0$ min), subsequent flight delay is only **12.43%** with a negative mean delay ($-2.05$ min).
   - Once preceding delay enters the **$30–44$ minute window**, the subsequent delay rate crosses the 50% tipping point (**50.84%**).
   - In severe delay categories ($>60$ min), subsequent delay probability stabilizes around **$60–62\%$**, while the *magnitude* of delay escalates steeply (mean delay reaches $+92.56$ min for preceding delays $\ge 120$ min).
2. **Transition Probability & Odds Ratio:**
   - $P(\text{Next Delayed} \mid \text{Prev Delayed}) = \mathbf{50.19\%}$
   - $P(\text{Next Delayed} \mid \text{Prev On-Time}) = \mathbf{13.97\%}$
   - **Relative Risk:** $\mathbf{3.59\times}$ | **Odds Ratio:** $\mathbf{6.20}$
   - Operating a delayed inbound flight increases the odds of subsequent outbound delay by over six-fold.


---
## 3. Turnaround Buffer Capacity & Deficit Dynamics

**Analytical Question:** How does the scheduled ground time act as an operational buffer, and what happens when incoming delay exceeds this buffer?


In [ ]:
# Display turnaround buffer bins
df_buf = pd.read_csv("results/tables/eda_turnaround_buffer_bins.csv")
display(df_buf)

# Display turnaround deficit impact
df_deficit = pd.read_csv("results/tables/eda_turnaround_deficit_impact.csv")
display(df_deficit)

# Display publication figure
display(Image(filename="results/figures/fig03_turnaround_buffer_dynamics.png"))


### Analytical Interpretation: Turnaround Buffer & Deficit Shock Absorption

1. **The Turnaround Buffer as a Physical Shock Absorber:**
   - In tight turns ($<45$ min scheduled buffer), delay propagation is **84.01%** if the preceding flight was delayed.
   - For standard turns ($60–74$ min), propagation drops to **64.48%**.
   - For generous turns ($90–119$ min), propagation drops to **42.84%**.
   - For overnight layovers ($\ge 180$ min), propagation drops to **19.24%** (below the baseline average).
2. **The Turnaround Deficit State:**
   - Under a Turnaround Deficit ($N = 382,937$, $5.66\%$ of pairs), the subsequent delay rate is **99.50%** with a mean delay of $+121.86$ minutes.
   - In non-deficit operations ($N = 6,385,550$, $94.34\%$ of pairs), the subsequent delay rate is **17.38%** with a mean delay of $+0.93$ minutes.
   - **Operational Framing:** Turnaround deficit is an empirical marker of buffer exhaustion. Downstream delays also reflect severe network congestion and adverse conditions, so it must be understood descriptively rather than as sole deterministic causality.


---
## 4. Temporal, Diurnal, and Seasonal Delay Patterns

**Analytical Question:** Do delay propagation patterns vary systematically across hours of the day, days of the week, and months of the year?


In [ ]:
# Display hourly diurnal patterns
df_hour = pd.read_csv("results/tables/eda_hourly_patterns.csv")
display(df_hour.head(10))

# Display publication figure
display(Image(filename="results/figures/fig04_temporal_delay_patterns.png"))


### Analytical Interpretation: Diurnal Accumulation & Temporal Seasonality

1. **Diurnal Snowball Effect:**
   - In early morning operations (05:00–07:00), incoming delays propagate at only **$11–17\%$**, because overnight buffers reset the system.
   - By 10:00, propagation crosses **51.76%**.
   - Between 16:00 and 19:00 (afternoon peak), propagation peaks at **$71.23\%$**, creating a 53-percentage-point divergence from on-time predecessor performance.
2. **Weekly & Monthly Seasonality:**
   - Peak delay months occur during summer travel (June–August, peaking at ~25% delay rate) and holiday travel (December/January).
   - Friday is the most delay-prone day of the week, while Tuesday and Saturday exhibit lower congestion.


---
## 5. Operating Carrier & Airport Operational Heterogeneity

**Analytical Question:** Does delay propagation sensitivity differ across airline operating models and hub turnaround environments?


In [ ]:
# Display top operating carriers
df_carrier = pd.read_csv("results/tables/eda_carrier_summary.csv")
display(df_carrier.head(10))

# Display top turnaround airports
df_ap = pd.read_csv("results/tables/eda_turnaround_airport_summary.csv")
display(df_ap.head(10))

# Display publication figure
display(Image(filename="results/figures/fig05_carrier_airport_heterogeneity.png"))


### Analytical Interpretation: Carrier & Airport Heterogeneity

1. **Carrier Operational Profiles:**
   - Network majors (Delta, United) exhibit lower overall deficit rates ($\sim 3.7\%$) and propagation rates ($40–47\%$), reflecting larger spare aircraft reserves and schedule padding.
   - Point-to-point and regional carriers (Southwest, SkyWest, PSA) exhibit tighter aircraft utilization and higher propagation rates ($51–59\%$).
   - Crucially, under a turnaround deficit, **all carriers suffer $>98.5\%$ delay rates**, demonstrating that buffer exhaustion overwhelms individual airline operational differences.
2. **Turnaround Hub Congestion:**
   - High-density hubs with severe runway constraints (ORD, DFW, DEN, CLT, EWR, LGA) experience elevated propagation rates ($>50\%$), whereas western fortress hubs with favourable weather and open capacity (SLC, PHX) demonstrate higher recovery resilience.


---
## 6. Aircraft Fleet Patterns and Delay Concentration

**Analytical Question:** Are delays distributed uniformly across the commercial fleet, or concentrated among a subset of heavily utilized aircraft?


In [ ]:
# Display consecutive delay progression
df_consec = pd.read_csv("results/tables/eda_aircraft_consecutive_delays.csv")
display(df_consec)

# Display fleet concentration metrics
df_conc = pd.read_csv("results/tables/eda_aircraft_concentration.csv")
display(df_conc)

# Display publication figure
display(Image(filename="results/figures/fig06_aircraft_level_patterns.png"))


### Analytical Interpretation: Aircraft Fleet Concentration

1. **Consecutive Delay Streak Accumulation:**
   - 0 prior delays today: Subsequent delay rate is **14.26%** (mean delay $-0.13$ min).
   - 1 prior delay today: Subsequent delay rate jumps to **62.54%** (mean delay $+47.44$ min).
   - 2 prior delays today: Subsequent delay rate reaches **71.43%** (mean delay $+58.61$ min).
   - 3+ prior delays today: Subsequent delay rate reaches **76.94%** (mean delay $+69.63$ min).
2. **Fleet Inequality (Gini Coefficient):**
   - The commercial fleet of **6,124 aircraft** exhibits a Gini coefficient of **0.2890** for accumulated delay minutes.
   - The top 20% of aircraft account for **34.67%** of all accumulated delay minutes, reflecting rotational scheduling variance rather than defective airframes.


---
## 7. Feature Relationships, Correlations, and Multicollinearity

**Analytical Question:** Which candidate predictors exhibit the strongest relationships with subsequent delay, and which present collinearity risks?


In [ ]:
# Display correlation table
df_corr = pd.read_csv("results/tables/eda_correlation_matrix.csv")
display(df_corr.sort_values(by="spearman_with_NEXT_ARR_DELAY", ascending=False))

# Display feature redundancy audit
df_red = pd.read_csv("results/tables/eda_feature_redundancy_audit.csv")
display(df_red)

# Display publication figure
display(Image(filename="results/figures/fig07_feature_correlations.png"))


### Analytical Interpretation: Correlation & Predictive Structure

1. **Top Predictive Features:**
   - `PREV_AIRCRAFT_CONSECUTIVE_DEL15` (Spearman $\rho = +0.4085$)
   - `IS_TURNAROUND_DEFICIT` (Pearson $r = +0.4909$, Spearman $\rho = +0.3779$)
   - `ESTIMATED_REMAINING_TURNAROUND_MINUTES` (Spearman $\rho = -0.3387$)
   - `PREV_ARR_DEL15` (Spearman $\rho = +0.3252$)
   - `PREV_ARR_DELAY` (Spearman $\rho = +0.3228$)
   - `NEXT_CRS_DEP_HOUR` (Spearman $\rho = +0.1778$)
2. **Multicollinearity & Redundancy Findings:**
   - `NEXT_DISTANCE` and `NEXT_CRS_ELAPSED_TIME` ($r = 0.981$) are virtually identical proxies for flight length.
   - `PREV_ARR_DELAY` and `PREV_DEP_DELAY` ($r = 0.884$) are highly collinear; `PREV_ARR_DELAY` should be preferred as it directly conditions gate turnaround.
   - Non-linear rank correlations (Spearman) are substantially stronger than linear correlations (Pearson) for remaining buffer, demonstrating the necessity of tree-based non-linear algorithms in Phase 6.


---
## 8. Summary of EDA Findings & Handoff to Phase 5

1. **Core Hypothesis Verified:** Preceding arrival delays provide substantial predictive power for subsequent flight delays ($RR = 3.59\times$, $OR = 6.20$).
2. **Buffer Exhaustion Mechanism:** Turnaround deficit leads to a $99.50\%$ subsequent delay rate, with buffer capacity acting as an effective non-linear shock absorber.
3. **Diurnal Network Buildup:** Propagation sensitivity escalates from $11\%$ at dawn to $71\%$ during evening peak hours.
4. **Modeling Guardrails for Phase 5 & 6:**
   - Chronological cross-validation is mandatory to prevent temporal data leakage.
   - Heavy-tailed target skewness dictates MAE/MedAE evaluation alongside RMSE.
   - Non-linear models (LightGBM, XGBoost) are strongly indicated over linear models due to buffer threshold dynamics.
